# High pLLPS Protein Interaction Analysis

This notebook analyzes whether high pLLPS proteins preferentially interact with other high pLLPS proteins.

**Workflow:**
1. Filter high pLLPS entries from the dataset
2. Fetch interaction partners from STRING database
3. Match interactors against the dataset to get their pLLPS scores
4. Test for high-high vs high-low over-representation

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
from scipy import stats

## Step 1: Load Data and Filter High pLLPS Proteins

In [ ]:
# Load the dataset
df = pd.read_excel('Human Phase separation data.xlsx')
print(f"Total proteins: {len(df)}")
print(f"p(LLPS) range: {df['p(LLPS)'].min():.3f} - {df['p(LLPS)'].max():.3f}")
df.head()

In [ ]:
# Define threshold for high pLLPS
PLLPS_THRESHOLD = 0.7

# Classify proteins
df['pLLPS_class'] = np.where(df['p(LLPS)'] >= PLLPS_THRESHOLD, 'high', 'low')

# Get high pLLPS proteins
high_pllps_df = df[df['pLLPS_class'] == 'high'].copy()
high_pllps_ids = high_pllps_df['Entry'].tolist()

print(f"High pLLPS proteins (>= {PLLPS_THRESHOLD}): {len(high_pllps_ids)} ({100*len(high_pllps_ids)/len(df):.1f}%)")
print(f"Low pLLPS proteins: {len(df) - len(high_pllps_ids)}")

## Step 2: Fetch Interaction Partners from STRING

In [ ]:
def fetch_interaction_partners(protein_ids, species=9606, score_threshold=700, batch_size=100):
    """
    Fetch interaction partners for a list of proteins from STRING.
    
    Parameters:
    - protein_ids: List of UniProt IDs
    - species: NCBI taxonomy ID (9606 = human)
    - score_threshold: Minimum confidence score (0-1000)
    - batch_size: Proteins per API request
    
    Returns:
    - DataFrame with interactions
    """
    string_api_url = "https://string-db.org/api/json/network"
    all_interactions = []
    
    total_batches = (len(protein_ids) + batch_size - 1) // batch_size
    print(f"Fetching interactions for {len(protein_ids)} proteins in {total_batches} batches...")
    
    for i in range(0, len(protein_ids), batch_size):
        batch = protein_ids[i:i+batch_size]
        batch_num = i // batch_size + 1
        
        params = {
            "identifiers": "\r".join(batch),
            "species": species,
            "required_score": score_threshold,
            "caller_identity": "pllps_analysis"
        }
        
        try:
            response = requests.post(string_api_url, data=params, timeout=60)
            if response.status_code == 200:
                interactions = response.json()
                all_interactions.extend(interactions)
                print(f"  Batch {batch_num}/{total_batches}: {len(interactions)} interactions")
            else:
                print(f"  Batch {batch_num}/{total_batches}: Error {response.status_code}")
        except Exception as e:
            print(f"  Batch {batch_num}/{total_batches}: {e}")
        
        time.sleep(1)  # Rate limiting
    
    if all_interactions:
        return pd.DataFrame(all_interactions)
    return pd.DataFrame()

In [ ]:
# Fetch interactions for high pLLPS proteins
# Note: Use a subset for testing, or all for full analysis
sample_size = min(100, len(high_pllps_ids))  # Adjust as needed
sample_proteins = high_pllps_ids[:sample_size]

print(f"Analyzing {sample_size} high pLLPS proteins...")
interactions_df = fetch_interaction_partners(sample_proteins, score_threshold=700)  # High confidence

In [ ]:
# View the interaction data
if len(interactions_df) > 0:
    print(f"Total interactions found: {len(interactions_df)}")
    print(f"Columns: {interactions_df.columns.tolist()}")
    interactions_df.head()

## Step 3: Match Interactors Against Dataset

In [ ]:
def match_interactors_to_pllps(interactions_df, pllps_df):
    """
    Match interaction partners to the pLLPS dataset.
    
    Returns DataFrame with both proteins' pLLPS values.
    """
    if len(interactions_df) == 0:
        return pd.DataFrame()
    
    # Create lookup dictionary: Entry -> pLLPS
    pllps_lookup = dict(zip(pllps_df['Entry'], pllps_df['p(LLPS)']))
    
    # Also create lookup by Entry name (STRING often uses gene names)
    entry_name_lookup = dict(zip(pllps_df['Entry name'], pllps_df['p(LLPS)']))
    entry_to_name = dict(zip(pllps_df['Entry'], pllps_df['Entry name']))
    
    # Extract protein names from STRING format
    # STRING preferredName might be gene names like 'TP53', 'BRCA1', etc.
    results = []
    
    for _, row in interactions_df.iterrows():
        protein_a = row.get('preferredName_A', row.get('stringId_A', ''))
        protein_b = row.get('preferredName_B', row.get('stringId_B', ''))
        score = row.get('score', 0)
        
        # Try to match proteins to pLLPS data
        pllps_a = pllps_lookup.get(protein_a) or entry_name_lookup.get(protein_a)
        pllps_b = pllps_lookup.get(protein_b) or entry_name_lookup.get(protein_b)
        
        results.append({
            'protein_a': protein_a,
            'protein_b': protein_b,
            'score': score,
            'pllps_a': pllps_a,
            'pllps_b': pllps_b
        })
    
    return pd.DataFrame(results)

In [ ]:
# Match interactors to pLLPS data
matched_df = match_interactors_to_pllps(interactions_df, df)

if len(matched_df) > 0:
    # Show match statistics
    matched_a = matched_df['pllps_a'].notna().sum()
    matched_b = matched_df['pllps_b'].notna().sum()
    both_matched = (matched_df['pllps_a'].notna() & matched_df['pllps_b'].notna()).sum()
    
    print(f"Total interactions: {len(matched_df)}")
    print(f"Protein A matched to pLLPS: {matched_a} ({100*matched_a/len(matched_df):.1f}%)")
    print(f"Protein B matched to pLLPS: {matched_b} ({100*matched_b/len(matched_df):.1f}%)")
    print(f"Both proteins matched: {both_matched} ({100*both_matched/len(matched_df):.1f}%)")
    
    matched_df.head(10)

## Step 4: Test for High-High vs High-Low Over-representation

In [ ]:
def analyze_interaction_enrichment(matched_df, pllps_threshold=0.7):
    """
    Analyze if high pLLPS proteins preferentially interact with each other.
    
    Tests:
    1. Count high-high, high-low, low-low interactions
    2. Compare to expected distribution (null hypothesis)
    3. Calculate enrichment ratio
    4. Perform chi-squared test
    """
    # Filter to interactions where both proteins have pLLPS scores
    complete_df = matched_df.dropna(subset=['pllps_a', 'pllps_b']).copy()
    
    if len(complete_df) == 0:
        print("No interactions with complete pLLPS data found.")
        return None
    
    # Classify each protein
    complete_df['class_a'] = np.where(complete_df['pllps_a'] >= pllps_threshold, 'high', 'low')
    complete_df['class_b'] = np.where(complete_df['pllps_b'] >= pllps_threshold, 'high', 'low')
    
    # Count interaction types
    def get_interaction_type(row):
        if row['class_a'] == 'high' and row['class_b'] == 'high':
            return 'high-high'
        elif row['class_a'] == 'low' and row['class_b'] == 'low':
            return 'low-low'
        else:
            return 'high-low'
    
    complete_df['interaction_type'] = complete_df.apply(get_interaction_type, axis=1)
    
    # Count each type
    counts = complete_df['interaction_type'].value_counts()
    total = len(complete_df)
    
    high_high = counts.get('high-high', 0)
    high_low = counts.get('high-low', 0)
    low_low = counts.get('low-low', 0)
    
    print(f"\n{'='*50}")
    print("INTERACTION TYPE ANALYSIS")
    print(f"{'='*50}")
    print(f"\nTotal interactions analyzed: {total}")
    print(f"\nObserved counts:")
    print(f"  High-High: {high_high} ({100*high_high/total:.1f}%)")
    print(f"  High-Low:  {high_low} ({100*high_low/total:.1f}%)")
    print(f"  Low-Low:   {low_low} ({100*low_low/total:.1f}%)")
    
    # Calculate expected distribution (null hypothesis)
    # Get proportion of high pLLPS among all proteins in interactions
    all_proteins = pd.concat([
        complete_df[['protein_a', 'pllps_a']].rename(columns={'protein_a': 'protein', 'pllps_a': 'pllps'}),
        complete_df[['protein_b', 'pllps_b']].rename(columns={'protein_b': 'protein', 'pllps_b': 'pllps'})
    ]).drop_duplicates(subset='protein')
    
    n_high = (all_proteins['pllps'] >= pllps_threshold).sum()
    n_total = len(all_proteins)
    p_high = n_high / n_total if n_total > 0 else 0
    p_low = 1 - p_high
    
    # Expected probabilities under random pairing
    expected_high_high = p_high * p_high
    expected_high_low = 2 * p_high * p_low
    expected_low_low = p_low * p_low
    
    print(f"\nBackground proportion of high pLLPS: {p_high:.3f} ({n_high}/{n_total})")
    print(f"\nExpected by chance:")
    print(f"  High-High: {expected_high_high*100:.1f}%")
    print(f"  High-Low:  {expected_high_low*100:.1f}%")
    print(f"  Low-Low:   {expected_low_low*100:.1f}%")
    
    # Calculate enrichment
    observed_high_high = high_high / total if total > 0 else 0
    enrichment = observed_high_high / expected_high_high if expected_high_high > 0 else 0
    
    print(f"\n{'='*50}")
    print(f"ENRICHMENT OF HIGH-HIGH INTERACTIONS: {enrichment:.2f}x")
    print(f"{'='*50}")
    
    # Chi-squared test
    observed = [high_high, high_low, low_low]
    expected = [expected_high_high * total, expected_high_low * total, expected_low_low * total]
    
    if all(e > 5 for e in expected):  # Chi-squared validity check
        chi2, p_value = stats.chisquare(observed, expected)
        print(f"\nChi-squared test:")
        print(f"  Chi² = {chi2:.2f}")
        print(f"  p-value = {p_value:.2e}")
        
        if p_value < 0.05:
            print(f"\n✅ Significant deviation from random expectation (p < 0.05)")
            if enrichment > 1:
                print(f"   High pLLPS proteins preferentially interact with other high pLLPS proteins!")
            else:
                print(f"   High pLLPS proteins avoid interacting with other high pLLPS proteins.")
        else:
            print(f"\n⚠️ No significant deviation from random expectation (p >= 0.05)")
    else:
        print(f"\n⚠️ Chi-squared test not valid (expected counts too small)")
    
    return {
        'high_high': high_high,
        'high_low': high_low,
        'low_low': low_low,
        'total': total,
        'enrichment': enrichment,
        'p_high': p_high,
        'complete_df': complete_df
    }

In [ ]:
# Run the enrichment analysis
results = analyze_interaction_enrichment(matched_df, pllps_threshold=PLLPS_THRESHOLD)

## Summary and Visualization

In [ ]:
# Visualize the results
if results is not None:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Pie chart of interaction types
    labels = ['High-High', 'High-Low', 'Low-Low']
    sizes = [results['high_high'], results['high_low'], results['low_low']]
    colors = ['#e74c3c', '#f39c12', '#3498db']
    
    axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    axes[0].set_title('Observed Interaction Types')
    
    # Bar chart comparing observed vs expected
    x = np.arange(3)
    width = 0.35
    
    total = results['total']
    p_high = results['p_high']
    p_low = 1 - p_high
    
    observed = [results['high_high']/total*100, results['high_low']/total*100, results['low_low']/total*100]
    expected = [p_high**2*100, 2*p_high*p_low*100, p_low**2*100]
    
    bars1 = axes[1].bar(x - width/2, observed, width, label='Observed', color='#2ecc71')
    bars2 = axes[1].bar(x + width/2, expected, width, label='Expected', color='#95a5a6')
    
    axes[1].set_ylabel('Percentage')
    axes[1].set_title('Observed vs Expected Interaction Distribution')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels)
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('pllps_interaction_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nFigure saved to: pllps_interaction_analysis.png")

## Conclusions

Based on the analysis:

1. **If enrichment > 1 and p < 0.05**: High pLLPS proteins preferentially interact with each other, suggesting they may co-localize in phase-separated condensates.

2. **If enrichment ≈ 1**: Interaction patterns follow random expectation - pLLPS doesn't influence interaction preferences.

3. **If enrichment < 1**: High pLLPS proteins may avoid each other, potentially indicating different functional roles or compartments.